In [12]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [13]:
import torch
import torch.nn as nn

# Q1 - creation of a simple transformer
model = nn.Transformer(d_model=512,
                        nhead=8,
                        num_encoder_layers=6,
                        num_decoder_layers=6)
print(model)

/home/shuptik/pw/sem6/ann/Transformers/.venv/lib/python3.12/site-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = TransformerEncoder(


Transformer(
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-5): 6 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
        )
        (linear1): Linear(in_features=512, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=2048, out_features=512, bias=True)
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
    (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (decoder): TransformerDecoder(
    (layers): ModuleList(
      (0-5): 6 x TransformerDecoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, o

In [14]:
import math
import torch
import torch.nn as nn

class TokenEmbedding(nn.Module):
    def __init__(self, embedding_dim: int, vocab_size: int) -> None:
        super().__init__()
        self.embedding_dim = embedding_dim
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.embedding(x) * math.sqrt(self.embedding_dim) # this is to properly scale token embedding wrt to positional embedding

In [15]:
import math
import torch.nn as nn
import torch

class PositionalEncoding(nn.Module):
    def __init__(self, block_size: int, embedding_dim: int) -> None:
        super().__init__()
        positional_embeddings = torch.zeros(block_size, embedding_dim)
        for pos in range(block_size):
            for index in range(embedding_dim // 2):
                denominator: float = 10000 ** (2 * index / embedding_dim)
                positional_embeddings[pos, 2 * index] = math.sin(pos / denominator)
                positional_embeddings[pos, 2 * index + 1] = math.cos(pos / denominator)
        self.register_buffer("positional_embeddings", positional_embeddings)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.positional_embeddings[:x.size(dim = 1), :]
    
vocab_size = 10000
embedding_dim = 512
sequence_length = 10
batch_size = 2
token_embedding = TokenEmbedding(embedding_dim=embedding_dim, vocab_size=vocab_size)
positional_embedding = PositionalEncoding(block_size=100, embedding_dim=embedding_dim)
id_tensor: torch.Tensor = torch.randint(low = 0, high=10, size = (batch_size, sequence_length))
result = positional_embedding(token_embedding(id_tensor))
print(result.shape)
print(result)

torch.Size([2, 10, 512])
tensor([[[  1.4729, -22.5645,   6.8399,  ...,  17.0795,  18.7411,  18.5407],
         [  2.3143, -23.0242,   7.6617,  ...,  17.0795,  18.7412,  18.5407],
         [ -8.4826,  23.2277, -27.7743,  ...,  -8.1922,  16.5021,  27.8460],
         ...,
         [ 28.8518,  20.7003,  12.6169,  ...,  -0.4417,  22.8311,  -1.1631],
         [ -5.4351, -16.6426,   5.6908,  ..., -13.2894, -21.8485,  24.2973],
         [ 57.5867,  10.1272,   8.9892,  ...,  -3.6063, -23.3441,  22.8481]],

        [[ -6.4244, -15.4971,   4.7001,  ..., -13.2894, -21.8493,  24.2973],
         [ -3.3601, -31.3965,   2.7813,  ...,   0.6754,  27.4272,   4.1948],
         [ 18.2916, -11.9185, -32.7645,  ..., -22.6875,  -7.0712,   7.3328],
         ...,
         [  2.5362,  34.6394,  -3.6921,  ...,  -2.7208, -12.8344,  13.6319],
         [  2.0670,   4.1409,  34.5770,  ..., -10.8980,  10.1591, -51.6715],
         [ -3.7895, -32.8479,   2.6358,  ...,   0.6754,  27.4280,   4.1948]]],
       grad_fn=<Add

In [16]:
import torch.nn.functional as F

class AttentionHead(nn.Module):
    def __init__(self, head_size: int) -> None:
        super().__init__()
        self.head_size = head_size
        self.query = nn.Linear(in_features=head_size, out_features=head_size, bias=False)
        self.key = nn.Linear(in_features = head_size, out_features=head_size, bias=False)
        self.values = nn.Linear(in_features = head_size, out_features=head_size, bias=False)
    def forward(self, x: torch.Tensor):
        (B, S, D) = x.shape
        q = self.query(x)
        k = self.key(x)
        v = self.values(x)
        weights =  F.softmax((q @ k.transpose(-2, -1))*self.head_size ** -0.5, dim = -1) # (B,S,S) shape
        scores = weights @ v # (B,S,D)
        return scores


class MultiHeadAttention(nn.Module):
    def __init__(self, n_heads: int, embedding_dim: int) -> None:
        super().__init__()
        self.n_heads = n_heads
        self.embedding_dim = embedding_dim
        self.head_dim = embedding_dim // n_heads
        self.attention_heads = nn.ModuleList(
            AttentionHead(self.head_dim)
            for _ in range(n_heads)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        head_outputs = []
        for index in range(self.n_heads):
            start = index * self.head_dim
            end = start + self.head_dim
            x_head = x[:, :, start:end]
            attention = self.attention_heads[index](x_head)
            head_outputs.append(attention)
        return torch.cat(head_outputs, dim=-1)
    
n_heads = 8
embedding_dim = 512
attention_block = MultiHeadAttention(n_heads=n_heads, embedding_dim=embedding_dim)
input = result
result = attention_block(input)
print(result.shape)
print(result.shape == input.shape)


torch.Size([2, 10, 512])
True


In [22]:
import torch.nn as nn
import torch

class FeedForward(nn.Module):
    def __init__(self, embedding_dim : int, dropout: float) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features=embedding_dim, out_features=4*embedding_dim),
            nn.ReLU(),
            nn.Linear(in_features=4 * embedding_dim, out_features=embedding_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

class EncoderLayer(nn.Module):
    def __init__(self, n_heads: int, embedding_dim: int, dropout: float) -> None:
        super().__init__()
        self.ln1 = nn.LayerNorm(embedding_dim)
        self.ln2 = nn.LayerNorm(embedding_dim)
        self.fwd = FeedForward(embedding_dim, dropout)
        self.ma = MultiHeadAttention(n_heads=n_heads, embedding_dim=embedding_dim)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.ma(self.ln1(x))
        x = x + self.fwd(self.ln2(x))
        return x

class ClassificationHead(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.linear = nn.Linear(in_features=in_features, out_features=out_features)
    
    def forward(self, x: torch.Tensor):
        return self.linear(x)
        


class TransformerEncoder(nn.Module):
    def __init__(self, layer_num : int, n_heads: int, embedding_dim: int, vocab_size: int, block_size: int, num_classes: int, dropout: float) -> None:
        super().__init__()
        
        self.embedding = TokenEmbedding(embedding_dim, vocab_size)
        self.pos_embedding = PositionalEncoding(block_size, embedding_dim)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.Sequential(*[EncoderLayer(n_heads, embedding_dim, dropout) for _ in range(layer_num)])
        self.ln = nn.LayerNorm(embedding_dim)
        self.ch = nn.Linear(in_features=embedding_dim, out_features=num_classes)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        idx = self.embedding(idx)
        idx = self.pos_embedding(idx)
        idx = self.drop(idx)
        idx = self.blocks(idx) 
        idx = self.ln(idx)
        idx = idx.mean(dim=1)  # [B, D]
        logits = self.ch(idx)
        return logits
    
layer_num = 6
n_heads = 8
num_classes = 3
embedding_dim = 512
block_size = 10
vocab_size = 10000
id_tensor: torch.Tensor = torch.randint(low = 0, high=vocab_size, size = (2, block_size - 1))
model = TransformerEncoder(layer_num, n_heads, embedding_dim, vocab_size, block_size, num_classes, dropout=0.4)
logits = model(id_tensor)
probs = F.softmax(logits, dim = -1)
result = probs.argmax(dim = -1)
print(result)
print(result.shape)
print(probs)
print(probs.shape)

tensor([0, 0])
torch.Size([2])
tensor([[0.4410, 0.2546, 0.3044],
        [0.3837, 0.3596, 0.2567]], grad_fn=<SoftmaxBackward0>)
torch.Size([2, 3])
